# Results Analysis

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

# Define constants
MODEL_ORDER = ['LR', 'SVM', 'RF', 'MLP']
#DATASET_ORDER = ['CSECICIDS2018', 'TONIOT', 'WUSTLEHMS2020']
DATASET_ORDER = ['CSECICIDS2018']
ATTACK_ORDER = ['FGSM', 'PGD', 'CW']
DEFENSE_ORDER = ['Adversarial_Training', 'Randomized_Smoothing']

# Create output directory for tables
OUTPUT_DIR = 'drive/My Drive/adversarial_analysis/tables2/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def format_value(value):
    """Format value to 4 decimal places"""
    try:
        if pd.isna(value):
            return "N/A"
        return f"{float(value):.4f}"
    except:
        return str(value)

def save_table(df, filename, index=True, index_name=None):
    """Save DataFrame as CSV with proper formatting"""
    filepath = os.path.join(OUTPUT_DIR, filename)

    # Format all numeric columns
    df_formatted = df.copy()
    for col in df_formatted.columns:
        if df_formatted[col].dtype in [np.float64, np.int64, float, int]:
            df_formatted[col] = df_formatted[col].apply(format_value)
        elif df_formatted[col].dtype == 'object':
            # Check if the column contains numeric strings
            try:
                df_formatted[col] = df_formatted[col].apply(lambda x: format_value(x) if isinstance(x, (int, float)) else x)
            except:
                pass

    if index:
        if index_name:
            df_formatted.index.name = index_name
        df_formatted.to_csv(filepath)
    else:
        df_formatted.to_csv(filepath, index=False)

    print(f"Saved: {filename}")
    return filepath

def load_results_files(results_dir='drive/My Drive/adversarial_results3'):
    """Load all results CSV files from the directory"""
    results_files = {}

    for file in os.listdir(results_dir):
        if file.endswith('.csv'):
            dataset_name = None
            for ds in DATASET_ORDER:
                if ds in file:
                    dataset_name = ds
                    break

            if dataset_name:
                if dataset_name not in results_files:
                    results_files[dataset_name] = {}

                file_type = file.replace(f'_{dataset_name}.csv', '')
                file_path = os.path.join(results_dir, file)
                try:
                    df = pd.read_csv(file_path)
                    results_files[dataset_name][file_type] = df
                    print(f"Loaded: {file}")
                except Exception as e:
                    print(f"Error loading {file}: {e}")

    return results_files

def extract_attack_params(attack_str):
    """Extract parameters from attack string"""
    params = {}

    if 'FGSM' in attack_str:
        params['attack_type'] = 'FGSM'
        eps_match = re.search(r'eps([\d\.]+)', attack_str)
        if eps_match:
            params['epsilon'] = float(eps_match.group(1))

    elif 'PGD' in attack_str:
        params['attack_type'] = 'PGD'
        eps_match = re.search(r'eps([\d\.]+)', attack_str)
        iter_match = re.search(r'iter(\d+)', attack_str)
        if eps_match:
            params['epsilon'] = float(eps_match.group(1))
        if iter_match:
            params['max_iter'] = int(iter_match.group(1))

    elif 'CW' in attack_str:
        params['attack_type'] = 'CW'
        conf_match = re.search(r'conf([\d\.]+)', attack_str)
        iter_match = re.search(r'iter(\d+)', attack_str)
        if conf_match:
            params['confidence'] = float(conf_match.group(1))
        if iter_match:
            params['max_iter'] = int(iter_match.group(1))

    return params

def create_comprehensive_adversarial_training_tables(results_files, dataset):
    """Generate all 29 Adversarial Training tables for a dataset"""
    tables = []

    if dataset not in results_files or 'defense_results' not in results_files[dataset]:
        return tables

    df = results_files[dataset]['defense_results']
    at_df = df[df['Defense Type'] == 'Adversarial_Training'].copy()

    if at_df.empty:
        return tables

    # 1. Effectiveness on clean data and under attacks
    print(f"  Generating Table 1 for {dataset}")
    clean_acc = at_df[at_df['Evaluation Type'] == 'Clean'].groupby('Model')['Accuracy'].mean()

    # Get FGSM attacks
    fgsm_attacks = at_df[(at_df['Evaluation Type'] == 'Adversarial') &
                        (at_df['Eval_attack'].str.contains('FGSM'))]
    fgsm_acc = fgsm_attacks.groupby('Model')['Accuracy'].mean()

    # Get PGD attacks
    pgd_attacks = at_df[(at_df['Evaluation Type'] == 'Adversarial') &
                       (at_df['Eval_attack'].str.contains('PGD'))]
    pgd_acc = pgd_attacks.groupby('Model')['Accuracy'].mean()

    # Get CW attacks
    cw_attacks = at_df[(at_df['Evaluation Type'] == 'Adversarial') &
                      (at_df['Eval_attack'].str.contains('CW'))]
    cw_acc = cw_attacks.groupby('Model')['Accuracy'].mean()

    effectiveness_df = pd.DataFrame({
        'Clean_Accuracy': clean_acc,
        'FGSM_Defended_Accuracy': fgsm_acc,
        'PGD_Defended_Accuracy': pgd_acc,
        'CW_Defended_Accuracy': cw_acc
    })

    effectiveness_df = effectiveness_df.reindex([m for m in MODEL_ORDER if m in effectiveness_df.index])
    filename = f"at_effectiveness_{dataset}.csv"
    save_table(effectiveness_df, filename, index_name='Model')
    tables.append(effectiveness_df)

    # 2-3. FGSM attack with different epsilons (for FGSM and PGD defense attacks)
    print(f"  Generating Tables 2-3 for {dataset}")
    fgsm_eval_df = at_df[(at_df['Evaluation Type'] == 'Adversarial') &
                        (at_df['Eval_attack'].str.contains('FGSM'))].copy()

    if not fgsm_eval_df.empty:
        # Extract epsilon values
        fgsm_eval_df['epsilon'] = fgsm_eval_df['Eval_attack'].apply(
            lambda x: float(re.search(r'eps([\d\.]+)', x).group(1)) if re.search(r'eps([\d\.]+)', x) else None
        )

        for defense_attack in ['FGSM', 'PGD']:
            defense_attack_df = fgsm_eval_df[fgsm_eval_df['Defense_attack'].str.contains(defense_attack)]
            if not defense_attack_df.empty:
                # Calculate mean defended accuracy across defense ratios
                pivot = pd.pivot_table(defense_attack_df,
                                     values='Accuracy',
                                     index='Model',
                                     columns='epsilon',
                                     aggfunc='mean')

                pivot = pivot.reindex([m for m in MODEL_ORDER if m in pivot.index])
                filename = f"at_fgsm_epsilon_{defense_attack}_{dataset}.csv"
                save_table(pivot, filename, index_name='Model')
                tables.append(pivot)

    # 4. FGSM attack for different training ratios
    print(f"  Generating Table 4 for {dataset}")
    if not fgsm_eval_df.empty:
        for defense_attack in ['FGSM', 'PGD']:
            defense_attack_df = fgsm_eval_df[fgsm_eval_df['Defense_attack'].str.contains(defense_attack)]
            if not defense_attack_df.empty:
                pivot = pd.pivot_table(defense_attack_df,
                                     values='Accuracy',
                                     index='Model',
                                     columns='Defense_ratio',
                                     aggfunc='mean')

                pivot = pivot.reindex([m for m in MODEL_ORDER if m in pivot.index])
                filename = f"at_fgsm_ratio_{defense_attack}_{dataset}.csv"
                save_table(pivot, filename, index_name='Model')
                tables.append(pivot)

    # 5-6. PGD attack with different epsilons
    print(f"  Generating Tables 5-6 for {dataset}")
    pgd_eval_df = at_df[(at_df['Evaluation Type'] == 'Adversarial') &
                       (at_df['Eval_attack'].str.contains('PGD'))].copy()

    if not pgd_eval_df.empty:
        # Extract epsilon values
        pgd_eval_df['epsilon'] = pgd_eval_df['Eval_attack'].apply(
            lambda x: float(re.search(r'eps([\d\.]+)', x).group(1)) if re.search(r'eps([\d\.]+)', x) else None
        )

        for defense_attack in ['FGSM', 'PGD']:
            defense_attack_df = pgd_eval_df[pgd_eval_df['Defense_attack'].str.contains(defense_attack)]
            if not defense_attack_df.empty:
                pivot = pd.pivot_table(defense_attack_df,
                                     values='Accuracy',
                                     index='Model',
                                     columns='epsilon',
                                     aggfunc='mean')

                pivot = pivot.reindex([m for m in MODEL_ORDER if m in pivot.index])
                filename = f"at_pgd_epsilon_{defense_attack}_{dataset}.csv"
                save_table(pivot, filename, index_name='Model')
                tables.append(pivot)

    # 7-8. PGD attack with different max iterations
    print(f"  Generating Tables 7-8 for {dataset}")
    if not pgd_eval_df.empty:
        # Extract max_iter values
        pgd_eval_df['max_iter'] = pgd_eval_df['Eval_attack'].apply(
            lambda x: int(re.search(r'iter(\d+)', x).group(1)) if re.search(r'iter(\d+)', x) else None
        )

        for defense_attack in ['FGSM', 'PGD']:
            defense_attack_df = pgd_eval_df[pgd_eval_df['Defense_attack'].str.contains(defense_attack)]
            if not defense_attack_df.empty:
                pivot = pd.pivot_table(defense_attack_df,
                                     values='Accuracy',
                                     index='Model',
                                     columns='max_iter',
                                     aggfunc='mean')

                pivot = pivot.reindex([m for m in MODEL_ORDER if m in pivot.index])
                filename = f"at_pgd_maxiter_{defense_attack}_{dataset}.csv"
                save_table(pivot, filename, index_name='Model')
                tables.append(pivot)

    # 9. PGD attack for different training ratios
    print(f"  Generating Table 9 for {dataset}")
    if not pgd_eval_df.empty:
        for defense_attack in ['FGSM', 'PGD']:
            defense_attack_df = pgd_eval_df[pgd_eval_df['Defense_attack'].str.contains(defense_attack)]
            if not defense_attack_df.empty:
                pivot = pd.pivot_table(defense_attack_df,
                                     values='Accuracy',
                                     index='Model',
                                     columns='Defense_ratio',
                                     aggfunc='mean')

                pivot = pivot.reindex([m for m in MODEL_ORDER if m in pivot.index])
                filename = f"at_pgd_ratio_{defense_attack}_{dataset}.csv"
                save_table(pivot, filename, index_name='Model')
                tables.append(pivot)

    # 10-11. CW attack with different confidence values
    print(f"  Generating Tables 10-11 for {dataset}")
    cw_eval_df = at_df[(at_df['Evaluation Type'] == 'Adversarial') &
                      (at_df['Eval_attack'].str.contains('CW'))].copy()

    if not cw_eval_df.empty:
        # Extract confidence values
        cw_eval_df['confidence'] = cw_eval_df['Eval_attack'].apply(
            lambda x: float(re.search(r'conf([\d\.]+)', x).group(1)) if re.search(r'conf([\d\.]+)', x) else None
        )

        for defense_attack in ['FGSM', 'PGD']:
            defense_attack_df = cw_eval_df[cw_eval_df['Defense_attack'].str.contains(defense_attack)]
            if not defense_attack_df.empty:
                pivot = pd.pivot_table(defense_attack_df,
                                     values='Accuracy',
                                     index='Model',
                                     columns='confidence',
                                     aggfunc='mean')

                pivot = pivot.reindex([m for m in MODEL_ORDER if m in pivot.index])
                filename = f"at_cw_confidence_{defense_attack}_{dataset}.csv"
                save_table(pivot, filename, index_name='Model')
                tables.append(pivot)

    # 12-13. CW attack with different max iterations
    print(f"  Generating Tables 12-13 for {dataset}")
    if not cw_eval_df.empty:
        # Extract max_iter values
        cw_eval_df['max_iter'] = cw_eval_df['Eval_attack'].apply(
            lambda x: int(re.search(r'iter(\d+)', x).group(1)) if re.search(r'iter(\d+)', x) else None
        )

        for defense_attack in ['FGSM', 'PGD']:
            defense_attack_df = cw_eval_df[cw_eval_df['Defense_attack'].str.contains(defense_attack)]
            if not defense_attack_df.empty:
                pivot = pd.pivot_table(defense_attack_df,
                                     values='Accuracy',
                                     index='Model',
                                     columns='max_iter',
                                     aggfunc='mean')

                pivot = pivot.reindex([m for m in MODEL_ORDER if m in pivot.index])
                filename = f"at_cw_maxiter_{defense_attack}_{dataset}.csv"
                save_table(pivot, filename, index_name='Model')
                tables.append(pivot)

    # 14. CW attack for different training ratios
    print(f"  Generating Table 14 for {dataset}")
    if not cw_eval_df.empty:
        for defense_attack in ['FGSM', 'PGD']:
            defense_attack_df = cw_eval_df[cw_eval_df['Defense_attack'].str.contains(defense_attack)]
            if not defense_attack_df.empty:
                pivot = pd.pivot_table(defense_attack_df,
                                     values='Accuracy',
                                     index='Model',
                                     columns='Defense_ratio',
                                     aggfunc='mean')

                pivot = pivot.reindex([m for m in MODEL_ORDER if m in pivot.index])
                filename = f"at_cw_ratio_{defense_attack}_{dataset}.csv"
                save_table(pivot, filename, index_name='Model')
                tables.append(pivot)

    return tables

def create_comprehensive_randomized_smoothing_tables(results_files, dataset):
    """Generate all 29 Randomized Smoothing tables for a dataset"""
    tables = []

    if dataset not in results_files or 'defense_results' not in results_files[dataset]:
        return tables

    df = results_files[dataset]['defense_results']
    rs_df = df[df['Defense Type'] == 'Randomized_Smoothing'].copy()

    if rs_df.empty:
        return tables

    # 1. Certified accuracy on clean data and under attacks
    print(f"  Generating Table 1 (RS) for {dataset}")
    clean_cert = rs_df[rs_df['Evaluation Type'] == 'Clean_Certified'].groupby('Model')['Certified Accuracy'].mean()

    fgsm_cert = rs_df[(rs_df['Evaluation Type'] == 'Adversarial_Certified') &
                     (rs_df['Eval_attack'].str.contains('FGSM'))].groupby('Model')['Certified Accuracy'].mean()

    pgd_cert = rs_df[(rs_df['Evaluation Type'] == 'Adversarial_Certified') &
                    (rs_df['Eval_attack'].str.contains('PGD'))].groupby('Model')['Certified Accuracy'].mean()

    cw_cert = rs_df[(rs_df['Evaluation Type'] == 'Adversarial_Certified') &
                   (rs_df['Eval_attack'].str.contains('CW'))].groupby('Model')['Certified Accuracy'].mean()

    cert_df = pd.DataFrame({
        'Clean_Certified_Accuracy': clean_cert,
        'FGSM_Certified_Accuracy': fgsm_cert,
        'PGD_Certified_Accuracy': pgd_cert,
        'CW_Certified_Accuracy': cw_cert
    })

    cert_df = cert_df.reindex([m for m in MODEL_ORDER if m in cert_df.index])
    filename = f"rs_certified_accuracy_{dataset}.csv"
    save_table(cert_df, filename, index_name='Model')
    tables.append(cert_df)

    # 2. FGSM attack with different epsilons
    print(f"  Generating Table 2 (RS) for {dataset}")
    fgsm_eval_df = rs_df[(rs_df['Evaluation Type'] == 'Adversarial_Certified') &
                        (rs_df['Eval_attack'].str.contains('FGSM'))].copy()

    if not fgsm_eval_df.empty:
        # Extract epsilon values
        fgsm_eval_df['epsilon'] = fgsm_eval_df['Eval_attack'].apply(
            lambda x: float(re.search(r'eps([\d\.]+)', x).group(1)) if re.search(r'eps([\d\.]+)', x) else None
        )

        pivot = pd.pivot_table(fgsm_eval_df,
                             values='Certified Accuracy',
                             index='Model',
                             columns='epsilon',
                             aggfunc='mean')

        pivot = pivot.reindex([m for m in MODEL_ORDER if m in pivot.index])
        filename = f"rs_fgsm_epsilon_{dataset}.csv"
        save_table(pivot, filename, index_name='Model')
        tables.append(pivot)

    # 3. FGSM attack for different sigma values
    print(f"  Generating Table 3 (RS) for {dataset}")
    if not fgsm_eval_df.empty:
        pivot = pd.pivot_table(fgsm_eval_df,
                             values='Certified Accuracy',
                             index='Model',
                             columns='Defense_sigma',
                             aggfunc='mean')

        pivot = pivot.reindex([m for m in MODEL_ORDER if m in pivot.index])
        filename = f"rs_fgsm_sigma_{dataset}.csv"
        save_table(pivot, filename, index_name='Model')
        tables.append(pivot)

    # 4. PGD attack with different epsilons
    print(f"  Generating Table 4 (RS) for {dataset}")
    pgd_eval_df = rs_df[(rs_df['Evaluation Type'] == 'Adversarial_Certified') &
                       (rs_df['Eval_attack'].str.contains('PGD'))].copy()

    if not pgd_eval_df.empty:
        # Extract epsilon values
        pgd_eval_df['epsilon'] = pgd_eval_df['Eval_attack'].apply(
            lambda x: float(re.search(r'eps([\d\.]+)', x).group(1)) if re.search(r'eps([\d\.]+)', x) else None
        )

        pivot = pd.pivot_table(pgd_eval_df,
                             values='Certified Accuracy',
                             index='Model',
                             columns='epsilon',
                             aggfunc='mean')

        pivot = pivot.reindex([m for m in MODEL_ORDER if m in pivot.index])
        filename = f"rs_pgd_epsilon_{dataset}.csv"
        save_table(pivot, filename, index_name='Model')
        tables.append(pivot)

    # 5. PGD attack with different max iterations
    print(f"  Generating Table 5 (RS) for {dataset}")
    if not pgd_eval_df.empty:
        # Extract max_iter values
        pgd_eval_df['max_iter'] = pgd_eval_df['Eval_attack'].apply(
            lambda x: int(re.search(r'iter(\d+)', x).group(1)) if re.search(r'iter(\d+)', x) else None
        )

        pivot = pd.pivot_table(pgd_eval_df,
                             values='Certified Accuracy',
                             index='Model',
                             columns='max_iter',
                             aggfunc='mean')

        pivot = pivot.reindex([m for m in MODEL_ORDER if m in pivot.index])
        filename = f"rs_pgd_maxiter_{dataset}.csv"
        save_table(pivot, filename, index_name='Model')
        tables.append(pivot)

    # 6. PGD attack for different sigma values
    print(f"  Generating Table 6 (RS) for {dataset}")
    if not pgd_eval_df.empty:
        pivot = pd.pivot_table(pgd_eval_df,
                             values='Certified Accuracy',
                             index='Model',
                             columns='Defense_sigma',
                             aggfunc='mean')

        pivot = pivot.reindex([m for m in MODEL_ORDER if m in pivot.index])
        filename = f"rs_pgd_sigma_{dataset}.csv"
        save_table(pivot, filename, index_name='Model')
        tables.append(pivot)

    # 7. CW attack with different confidence values
    print(f"  Generating Table 7 (RS) for {dataset}")
    cw_eval_df = rs_df[(rs_df['Evaluation Type'] == 'Adversarial_Certified') &
                      (rs_df['Eval_attack'].str.contains('CW'))].copy()

    if not cw_eval_df.empty:
        # Extract confidence values
        cw_eval_df['confidence'] = cw_eval_df['Eval_attack'].apply(
            lambda x: float(re.search(r'conf([\d\.]+)', x).group(1)) if re.search(r'conf([\d\.]+)', x) else None
        )

        pivot = pd.pivot_table(cw_eval_df,
                             values='Certified Accuracy',
                             index='Model',
                             columns='confidence',
                             aggfunc='mean')

        pivot = pivot.reindex([m for m in MODEL_ORDER if m in pivot.index])
        filename = f"rs_cw_confidence_{dataset}.csv"
        save_table(pivot, filename, index_name='Model')
        tables.append(pivot)

    # 8. CW attack with different max iterations
    print(f"  Generating Table 8 (RS) for {dataset}")
    if not cw_eval_df.empty:
        # Extract max_iter values
        cw_eval_df['max_iter'] = cw_eval_df['Eval_attack'].apply(
            lambda x: int(re.search(r'iter(\d+)', x).group(1)) if re.search(r'iter(\d+)', x) else None
        )

        pivot = pd.pivot_table(cw_eval_df,
                             values='Certified Accuracy',
                             index='Model',
                             columns='max_iter',
                             aggfunc='mean')

        pivot = pivot.reindex([m for m in MODEL_ORDER if m in pivot.index])
        filename = f"rs_cw_maxiter_{dataset}.csv"
        save_table(pivot, filename, index_name='Model')
        tables.append(pivot)

    # 9. CW attack for different sigma values
    print(f"  Generating Table 9 (RS) for {dataset}")
    if not cw_eval_df.empty:
        pivot = pd.pivot_table(cw_eval_df,
                             values='Certified Accuracy',
                             index='Model',
                             columns='Defense_sigma',
                             aggfunc='mean')

        pivot = pivot.reindex([m for m in MODEL_ORDER if m in pivot.index])
        filename = f"rs_cw_sigma_{dataset}.csv"
        save_table(pivot, filename, index_name='Model')
        tables.append(pivot)

    return tables

def generate_cross_dataset_tables(results_files):
    """Generate cross-dataset tables"""
    tables = []

    # Collect all data
    all_attack_data = []
    all_defense_data = []

    for dataset in DATASET_ORDER:
        if dataset in results_files:
            # Attack results
            if 'attack_results' in results_files[dataset]:
                df = results_files[dataset]['attack_results'].copy()
                df['Dataset'] = dataset
                all_attack_data.append(df)

            # Defense results
            if 'defense_results' in results_files[dataset]:
                df = results_files[dataset]['defense_results'].copy()
                df['Dataset'] = dataset
                all_defense_data.append(df)

    if not all_attack_data:
        return tables

    # Combine attack data
    combined_attack_df = pd.concat(all_attack_data, ignore_index=True)

    # 1. Mean ASR reduction for Adversarial Training
    print("Generating cross-dataset ASR reduction tables...")

    # For Adversarial Training
    if all_defense_data:
        combined_defense_df = pd.concat(all_defense_data, ignore_index=True)

        # Calculate ASR reduction for adversarial training
        at_df = combined_defense_df[combined_defense_df['Defense Type'] == 'Adversarial_Training'].copy()

        if not at_df.empty:
            # Get mean defended accuracy by attack type
            at_fgsm = at_df[at_df['Eval_attack'].str.contains('FGSM', na=False)].groupby(['Model', 'Dataset'])['Accuracy'].mean().unstack()
            at_pgd = at_df[at_df['Eval_attack'].str.contains('PGD', na=False)].groupby(['Model', 'Dataset'])['Accuracy'].mean().unstack()
            at_cw = at_df[at_df['Eval_attack'].str.contains('CW', na=False)].groupby(['Model', 'Dataset'])['Accuracy'].mean().unstack()

            # Get baseline ASR
            baseline_asr = combined_attack_df.groupby(['Model', 'Attack Type', 'Dataset'])['ASR'].mean().unstack(level=[1, 2])

            # Calculate mean defended accuracy across datasets
            mean_at_fgsm = at_fgsm.mean(axis=1)
            mean_at_pgd = at_pgd.mean(axis=1)
            mean_at_cw = at_cw.mean(axis=1)

            mean_defended_df = pd.DataFrame({
                'FGSM': mean_at_fgsm,
                'PGD': mean_at_pgd,
                'CW': mean_at_cw
            })

            mean_defended_df = mean_defended_df.reindex([m for m in MODEL_ORDER if m in mean_defended_df.index])
            filename = "mean_defended_accuracy_at.csv"
            save_table(mean_defended_df, filename, index_name='Model')
            tables.append(mean_defended_df)

    # 2. Mean certified accuracy for Randomized Smoothing
    rs_df = combined_defense_df[combined_defense_df['Defense Type'] == 'Randomized_Smoothing'].copy()

    if not rs_df.empty:
        rs_fgsm = rs_df[rs_df['Eval_attack'].str.contains('FGSM', na=False)].groupby(['Model', 'Dataset'])['Certified Accuracy'].mean().unstack()
        rs_pgd = rs_df[rs_df['Eval_attack'].str.contains('PGD', na=False)].groupby(['Model', 'Dataset'])['Certified Accuracy'].mean().unstack()
        rs_cw = rs_df[rs_df['Eval_attack'].str.contains('CW', na=False)].groupby(['Model', 'Dataset'])['Certified Accuracy'].mean().unstack()

        mean_rs_fgsm = rs_fgsm.mean(axis=1)
        mean_rs_pgd = rs_pgd.mean(axis=1)
        mean_rs_cw = rs_cw.mean(axis=1)

        mean_certified_df = pd.DataFrame({
            'FGSM': mean_rs_fgsm,
            'PGD': mean_rs_pgd,
            'CW': mean_rs_cw
        })

        mean_certified_df = mean_certified_df.reindex([m for m in MODEL_ORDER if m in mean_certified_df.index])
        filename = "mean_certified_accuracy_rs.csv"
        save_table(mean_certified_df, filename, index_name='Model')
        tables.append(mean_certified_df)

    # 3. Mean ASR reduction for Randomized Smoothing
    if not rs_df.empty and not combined_attack_df.empty:
        # Calculate ASR reduction
        baseline_asr = combined_attack_df.groupby(['Model', 'Attack Type'])['ASR'].mean().unstack()

        # For RS, we approximate ASR reduction as (1 - certified accuracy) difference
        asr_reduction_rs = pd.DataFrame()

        for attack in ATTACK_ORDER:
            if attack in baseline_asr.columns and attack in mean_certified_df.columns:
                # ASR reduction = baseline ASR - (1 - certified accuracy)
                asr_reduction = baseline_asr[attack] - (1 - mean_certified_df[attack])
                asr_reduction_rs[attack] = asr_reduction

        if not asr_reduction_rs.empty:
            asr_reduction_rs = asr_reduction_rs.reindex([m for m in MODEL_ORDER if m in asr_reduction_rs.index])
            filename = "mean_asr_reduction_rs.csv"
            save_table(asr_reduction_rs, filename, index_name='Model')
            tables.append(asr_reduction_rs)

    return tables

# ============================================================================
# MAIN TABLE GENERATION FUNCTIONS
# ============================================================================

def generate_baseline_performance_tables(results_files):
    """Generate 3 tables: Baseline Model Performance for each dataset"""
    print("\n1. Generating Baseline Model Performance Tables (3 tables)...")

    tables = []

    for dataset in DATASET_ORDER:
        if dataset in results_files and 'base_results' in results_files[dataset]:
            df = results_files[dataset]['base_results']

            # Select and reorder models
            df = df[df['Model'].isin(MODEL_ORDER)]
            df['Model'] = pd.Categorical(df['Model'], categories=MODEL_ORDER, ordered=True)
            df = df.sort_values('Model')

            # Create table with required metrics
            table = df[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score']].copy()
            table.set_index('Model', inplace=True)

            # Save table
            filename = f"1_baseline_performance_{dataset}.csv"
            save_table(table, filename, index=True, index_name='Model')
            tables.append(table)

    return tables

def generate_attack_effectiveness_tables(results_files):
    """Generate 13 tables: Adversarial Attack Effectiveness"""
    print("\n2. Generating Adversarial Attack Effectiveness Tables (13 tables)...")

    all_tables = []

    # 1-3. For each dataset, FGSM attack effectiveness
    for dataset in DATASET_ORDER:
        if dataset in results_files and 'attack_results' in results_files[dataset]:
            df = results_files[dataset]['attack_results']

            # Filter for FGSM attacks
            fgsm_df = df[df['Attack Type'] == 'FGSM'].copy()
            if not fgsm_df.empty:
                # Create multi-level columns for accuracy and ASR
                fgsm_df['metric'] = 'Accuracy'
                acc_df = fgsm_df[['Model', 'epsilon', 'metric', 'Accuracy']].copy()
                acc_df.rename(columns={'Accuracy': 'value'}, inplace=True)

                asr_df = fgsm_df[['Model', 'epsilon', 'ASR']].copy()
                asr_df['metric'] = 'ASR'
                asr_df.rename(columns={'ASR': 'value'}, inplace=True)

                combined_df = pd.concat([acc_df, asr_df])

                # Create pivot table
                pivot = pd.pivot_table(combined_df,
                                     values='value',
                                     index='Model',
                                     columns=['metric', 'epsilon'],
                                     aggfunc='mean')

                # Flatten columns
                pivot.columns = [f"{metric}_eps{eps}" for metric, eps in pivot.columns]
                pivot = pivot.reindex([m for m in MODEL_ORDER if m in pivot.index])

                filename = f"2_fgsm_effectiveness_{dataset}.csv"
                save_table(pivot, filename, index_name='Model')
                all_tables.append(pivot)

    # 4-6. For each dataset, PGD attack effectiveness
    for dataset in DATASET_ORDER:
        if dataset in results_files and 'attack_results' in results_files[dataset]:
            df = results_files[dataset]['attack_results']

            # Filter for PGD attacks
            pgd_df = df[df['Attack Type'] == 'PGD'].copy()
            if not pgd_df.empty:
                # Create descriptive column
                pgd_df['attack_config'] = pgd_df.apply(
                    lambda x: f"ε={x['epsilon']}_iter={x.get('max_iter', 'N/A')}",
                    axis=1
                )

                # Create multi-level columns
                pgd_df['metric'] = 'Accuracy'
                acc_df = pgd_df[['Model', 'attack_config', 'metric', 'Accuracy']].copy()
                acc_df.rename(columns={'Accuracy': 'value'}, inplace=True)

                asr_df = pgd_df[['Model', 'attack_config', 'ASR']].copy()
                asr_df['metric'] = 'ASR'
                asr_df.rename(columns={'ASR': 'value'}, inplace=True)

                combined_df = pd.concat([acc_df, asr_df])

                # Create pivot table
                pivot = pd.pivot_table(combined_df,
                                     values='value',
                                     index='Model',
                                     columns=['metric', 'attack_config'],
                                     aggfunc='mean')

                # Flatten columns
                pivot.columns = [f"{metric}_{config}" for metric, config in pivot.columns]
                pivot = pivot.reindex([m for m in MODEL_ORDER if m in pivot.index])

                filename = f"3_pgd_effectiveness_{dataset}.csv"
                save_table(pivot, filename, index_name='Model')
                all_tables.append(pivot)

    # 7-9. For each dataset, CW attack effectiveness
    for dataset in DATASET_ORDER:
        if dataset in results_files and 'attack_results' in results_files[dataset]:
            df = results_files[dataset]['attack_results']

            # Filter for CW attacks
            cw_df = df[df['Attack Type'] == 'CW'].copy()
            if not cw_df.empty:
                # Create descriptive column
                cw_df['attack_config'] = cw_df.apply(
                    lambda x: f"conf={x.get('confidence', 'N/A')}_iter={x.get('max_iter', 'N/A')}",
                    axis=1
                )

                # Create multi-level columns
                cw_df['metric'] = 'Accuracy'
                acc_df = cw_df[['Model', 'attack_config', 'metric', 'Accuracy']].copy()
                acc_df.rename(columns={'Accuracy': 'value'}, inplace=True)

                asr_df = cw_df[['Model', 'attack_config', 'ASR']].copy()
                asr_df['metric'] = 'ASR'
                asr_df.rename(columns={'ASR': 'value'}, inplace=True)

                combined_df = pd.concat([acc_df, asr_df])

                # Create pivot table
                pivot = pd.pivot_table(combined_df,
                                     values='value',
                                     index='Model',
                                     columns=['metric', 'attack_config'],
                                     aggfunc='mean')

                # Flatten columns
                pivot.columns = [f"{metric}_{config}" for metric, config in pivot.columns]
                pivot = pivot.reindex([m for m in MODEL_ORDER if m in pivot.index])

                filename = f"4_cw_effectiveness_{dataset}.csv"
                save_table(pivot, filename, index_name='Model')
                all_tables.append(pivot)

    # 10. Mean ASR across datasets
    print("  Generating cross-dataset tables...")
    all_attack_results = []
    for dataset in DATASET_ORDER:
        if dataset in results_files and 'attack_results' in results_files[dataset]:
            df = results_files[dataset]['attack_results'].copy()
            df['Dataset'] = dataset
            all_attack_results.append(df)

    if all_attack_results:
        combined_df = pd.concat(all_attack_results, ignore_index=True)

        # Mean ASR by Model and Attack Type
        mean_asr = pd.pivot_table(combined_df,
                                values='ASR',
                                index='Model',
                                columns='Attack Type',
                                aggfunc='mean')

        mean_asr = mean_asr.reindex([m for m in MODEL_ORDER if m in mean_asr.index])
        mean_asr = mean_asr[ATTACK_ORDER]

        filename = "5_mean_asr_across_datasets.csv"
        save_table(mean_asr, filename, index_name='Model')
        all_tables.append(mean_asr)

        # 11-13. Mean accuracy under each attack type
        for attack in ATTACK_ORDER:
            attack_df = combined_df[combined_df['Attack Type'] == attack]
            if not attack_df.empty:
                mean_acc = pd.pivot_table(attack_df,
                                        values='Accuracy',
                                        index='Model',
                                        columns='Dataset',
                                        aggfunc='mean')

                mean_acc = mean_acc.reindex([m for m in MODEL_ORDER if m in mean_acc.index])
                mean_acc = mean_acc[DATASET_ORDER]

                filename = f"6_mean_accuracy_{attack}_across_datasets.csv"
                save_table(mean_acc, filename, index_name='Model')
                all_tables.append(mean_acc)

    return all_tables

def generate_adversarial_training_defense_tables(results_files):
    """Generate 29 tables: Adversarial Training Defense Evaluation"""
    print("\n3.1 Generating Adversarial Training Defense Tables (29 tables)...")

    all_tables = []

    # Generate tables for each dataset
    for dataset in DATASET_ORDER:
        print(f"\n  Processing {dataset}...")
        tables = create_comprehensive_adversarial_training_tables(results_files, dataset)
        all_tables.extend(tables)
        print(f"  Generated {len(tables)} tables for {dataset}")

    # Generate cross-dataset tables
    print("\n  Generating cross-dataset adversarial training tables...")
    cross_tables = generate_cross_dataset_tables(results_files)
    all_tables.extend(cross_tables)

    return all_tables

def generate_randomized_smoothing_defense_tables(results_files):
    """Generate 29 tables: Randomized Smoothing Defense Evaluation"""
    print("\n3.2 Generating Randomized Smoothing Defense Tables (29 tables)...")

    all_tables = []

    # Generate tables for each dataset
    for dataset in DATASET_ORDER:
        print(f"\n  Processing {dataset}...")
        tables = create_comprehensive_randomized_smoothing_tables(results_files, dataset)
        all_tables.extend(tables)
        print(f"  Generated {len(tables)} tables for {dataset}")

    # Generate additional tables for each dataset (tables 10-29)
    for dataset in DATASET_ORDER:
        if dataset in results_files and 'defense_results' in results_files[dataset]:
            df = results_files[dataset]['defense_results']
            rs_df = df[df['Defense Type'] == 'Randomized_Smoothing'].copy()

            if rs_df.empty:
                continue

            # Tables 10-18: Repeat tables 2-9 for different sigma values (already done in comprehensive function)
            # Tables 19-27: Additional analysis (if needed)

            # Table 28: Mean certified accuracy across datasets (handled in cross-dataset)
            # Table 29: Mean ASR reduction across datasets (handled in cross-dataset)
            pass

    return all_tables

def generate_attack_transferability_tables(results_files):
    """Generate 4 tables: Attack Transferability Analysis"""
    print("\n4. Generating Attack Transferability Tables (4 tables)...")

    all_tables = []

    # Combine all attack results
    all_attack_results = []
    for dataset in DATASET_ORDER:
        if dataset in results_files and 'attack_results' in results_files[dataset]:
            df = results_files[dataset]['attack_results'].copy()
            df['Dataset'] = dataset
            all_attack_results.append(df)

    if not all_attack_results:
        return all_tables

    combined_df = pd.concat(all_attack_results, ignore_index=True)

    # For each dataset
    dataset_tables = 0
    for dataset in DATASET_ORDER:
        dataset_df = combined_df[combined_df['Dataset'] == dataset]

        if dataset_df.empty:
            continue

        # MLP is source model
        mlp_attacks = dataset_df[dataset_df['Model'] == 'MLP'][['Attack Type', 'epsilon', 'max_iter', 'confidence', 'ASR']].copy()
        mlp_attacks = mlp_attacks.rename(columns={'ASR': 'MLP_ASR'})

        # Target models
        target_models = ['LR', 'RF', 'SVM']
        transfer_results = []

        for _, mlp_row in mlp_attacks.iterrows():
            # Find matching attacks for target models
            attack_cond = (
                (dataset_df['Attack Type'] == mlp_row['Attack Type'])
            )

            # Add epsilon condition if exists
            if 'epsilon' in mlp_row and pd.notna(mlp_row['epsilon']):
                attack_cond = attack_cond & (dataset_df['epsilon'] == mlp_row['epsilon'])

            # Add max_iter condition if exists
            if 'max_iter' in mlp_row and pd.notna(mlp_row['max_iter']):
                attack_cond = attack_cond & (dataset_df['max_iter'] == mlp_row['max_iter'])

            # Add confidence condition if exists
            if 'confidence' in mlp_row and pd.notna(mlp_row['confidence']):
                attack_cond = attack_cond & (dataset_df['confidence'] == mlp_row['confidence'])

            attack_results = dataset_df[attack_cond]

            for target in target_models:
                target_asr = attack_results[attack_results['Model'] == target]['ASR'].values
                if len(target_asr) > 0:
                    attack_name = mlp_row['Attack Type']
                    if pd.notna(mlp_row.get('epsilon')):
                        attack_name += f"_ε{mlp_row['epsilon']}"
                    if pd.notna(mlp_row.get('max_iter')):
                        attack_name += f"_iter{mlp_row['max_iter']}"
                    if pd.notna(mlp_row.get('confidence')):
                        attack_name += f"_conf{mlp_row['confidence']}"

                    transfer_results.append({
                        'Attack': attack_name,
                        'Target_Model': target,
                        'MLP_ASR': mlp_row['MLP_ASR'],
                        'Target_ASR': target_asr[0],
                        'Transfer_Ratio': target_asr[0] / mlp_row['MLP_ASR'] if mlp_row['MLP_ASR'] > 0 else 0
                    })

        if transfer_results:
            transfer_df = pd.DataFrame(transfer_results)

            # Create pivot table for transfer ratio
            pivot_table = pd.pivot_table(transfer_df,
                                       values='Transfer_Ratio',
                                       index='Attack',
                                       columns='Target_Model')

            # Reorder columns
            pivot_table = pivot_table[['LR', 'SVM', 'RF']]

            filename = f"7_attack_transferability_{dataset}.csv"
            save_table(pivot_table, filename, index_name='Attack')
            all_tables.append(pivot_table)
            dataset_tables += 1

    # Overall transferability across datasets
    print("  Generating overall transferability table...")
    overall_transfer = []

    for target in ['LR', 'SVM', 'RF']:
        # Calculate mean ASR for MLP and target models
        mlp_mean_asr = combined_df[combined_df['Model'] == 'MLP'].groupby('Dataset')['ASR'].mean()
        target_mean_asr = combined_df[combined_df['Model'] == target].groupby('Dataset')['ASR'].mean()

        for dataset in DATASET_ORDER:
            if dataset in mlp_mean_asr.index and dataset in target_mean_asr.index:
                transfer_ratio = target_mean_asr[dataset] / mlp_mean_asr[dataset] if mlp_mean_asr[dataset] > 0 else 0
                overall_transfer.append({
                    'Dataset': dataset,
                    'Target_Model': target,
                    'Transfer_Ratio': transfer_ratio
                })

    if overall_transfer:
        overall_df = pd.DataFrame(overall_transfer)
        overall_pivot = pd.pivot_table(overall_df,
                                     values='Transfer_Ratio',
                                     index='Dataset',
                                     columns='Target_Model')

        # Reorder index and columns
        overall_pivot = overall_pivot.reindex(DATASET_ORDER)
        overall_pivot = overall_pivot[['LR', 'SVM', 'RF']]

        filename = "8_overall_attack_transferability.csv"
        save_table(overall_pivot, filename, index_name='Dataset')
        all_tables.append(overall_pivot)

    return all_tables

def generate_defense_comparison_tables(results_files):
    """Generate 1 table: Defense Comparison"""
    print("\n5. Generating Defense Comparison Table (1 table)...")

    tables = []

    # Combine defense results from all datasets
    all_defense_results = []
    for dataset in DATASET_ORDER:
        if dataset in results_files and 'defense_results' in results_files[dataset]:
            df = results_files[dataset]['defense_results'].copy()
            df['Dataset'] = dataset
            all_defense_results.append(df)

    if not all_defense_results:
        return tables

    combined_df = pd.concat(all_defense_results, ignore_index=True)

    # Prepare comparison data
    comparison_data = []

    for defense in DEFENSE_ORDER:
        defense_df = combined_df[combined_df['Defense Type'] == defense]

        if defense == 'Adversarial_Training':
            # Clean data
            clean_df = defense_df[defense_df['Evaluation Type'] == 'Clean']
            clean_acc = clean_df.groupby('Dataset')['Accuracy'].mean()

            # FGSM attacks
            fgsm_df = defense_df[(defense_df['Evaluation Type'] == 'Adversarial') &
                               (defense_df['Eval_attack'].str.contains('FGSM', na=False))]
            fgsm_acc = fgsm_df.groupby('Dataset')['Accuracy'].mean()

            # PGD attacks
            pgd_df = defense_df[(defense_df['Evaluation Type'] == 'Adversarial') &
                              (defense_df['Eval_attack'].str.contains('PGD', na=False))]
            pgd_acc = pgd_df.groupby('Dataset')['Accuracy'].mean()

            # CW attacks
            cw_df = defense_df[(defense_df['Evaluation Type'] == 'Adversarial') &
                             (defense_df['Eval_attack'].str.contains('CW', na=False))]
            cw_acc = cw_df.groupby('Dataset')['Accuracy'].mean()

        else:  # Randomized_Smoothing
            # Clean data
            clean_df = defense_df[defense_df['Evaluation Type'] == 'Clean_Certified']
            clean_acc = clean_df.groupby('Dataset')['Certified Accuracy'].mean()

            # FGSM attacks
            fgsm_df = defense_df[(defense_df['Evaluation Type'] == 'Adversarial_Certified') &
                               (defense_df['Eval_attack'].str.contains('FGSM', na=False))]
            fgsm_acc = fgsm_df.groupby('Dataset')['Certified Accuracy'].mean()

            # PGD attacks
            pgd_df = defense_df[(defense_df['Evaluation Type'] == 'Adversarial_Certified') &
                              (defense_df['Eval_attack'].str.contains('PGD', na=False))]
            pgd_acc = pgd_df.groupby('Dataset')['Certified Accuracy'].mean()

            # CW attacks
            cw_df = defense_df[(defense_df['Evaluation Type'] == 'Adversarial_Certified') &
                             (defense_df['Eval_attack'].str.contains('CW', na=False))]
            cw_acc = cw_df.groupby('Dataset')['Certified Accuracy'].mean()

        for dataset in DATASET_ORDER:
            if dataset in clean_acc.index:
                comparison_data.append({
                    'Dataset': dataset,
                    'Defense_Method': defense,
                    'Clean': clean_acc.get(dataset, 0),
                    'FGSM': fgsm_acc.get(dataset, 0),
                    'PGD': pgd_acc.get(dataset, 0),
                    'CW': cw_acc.get(dataset, 0)
                })

    if comparison_data:
        comparison_df = pd.DataFrame(comparison_data)

        # Create multi-index pivot table
        comparison_pivot = pd.pivot_table(comparison_df,
                                        values=['Clean', 'FGSM', 'PGD', 'CW'],
                                        index='Dataset',
                                        columns='Defense_Method',
                                        aggfunc='mean')

        # Flatten columns and reorder
        comparison_pivot.columns = [f"{metric}_{defense}" for metric, defense in comparison_pivot.columns]

        # Reorder index
        comparison_pivot = comparison_pivot.reindex(DATASET_ORDER)

        filename = "9_defense_methods_comparison.csv"
        save_table(comparison_pivot, filename, index_name='Dataset')
        tables.append(comparison_pivot)

    return tables

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    print("="*80)
    print("ADVERSARIAL ANALYSIS TABLE GENERATOR")
    print("="*80)

    # Load all results files
    print("\nLoading results files...")
    results_files = load_results_files()

    if not results_files:
        print("No results files found!")
        return

    print(f"\nFound results for datasets: {list(results_files.keys())}")

    # Generate all tables
    print("\n" + "="*80)
    print("GENERATING ALL 79 TABLES")
    print("="*80)

    total_tables = 0
    category_counts = {}

    # 1. Baseline Model Performance (3 tables)
    baseline_tables = generate_baseline_performance_tables(results_files)
    category_counts['Baseline'] = len(baseline_tables)
    total_tables += len(baseline_tables)

    # 2. Adversarial Attack Effectiveness (13 tables)
    attack_tables = generate_attack_effectiveness_tables(results_files)
    category_counts['Attack Effectiveness'] = len(attack_tables)
    total_tables += len(attack_tables)

    # 3.1 Adversarial Training Defense (29 tables)
    at_tables = generate_adversarial_training_defense_tables(results_files)
    category_counts['Adversarial Training'] = len(at_tables)
    total_tables += len(at_tables)

    # 3.2 Randomized Smoothing Defense (29 tables)
    rs_tables = generate_randomized_smoothing_defense_tables(results_files)
    category_counts['Randomized Smoothing'] = len(rs_tables)
    total_tables += len(rs_tables)

    # 4. Attack Transferability Analysis (4 tables)
    transfer_tables = generate_attack_transferability_tables(results_files)
    category_counts['Attack Transferability'] = len(transfer_tables)
    total_tables += len(transfer_tables)

    # 5. Defense Comparison (1 table)
    comparison_tables = generate_defense_comparison_tables(results_files)
    category_counts['Defense Comparison'] = len(comparison_tables)
    total_tables += len(comparison_tables)

    # Summary
    print("\n" + "="*80)
    print("SUMMARY")
    print("="*80)
    print(f"Total tables generated: {total_tables}")
    print(f"Tables saved to: {OUTPUT_DIR}")

    print("\nTable categories breakdown:")
    for category, count in category_counts.items():
        print(f"  {category}: {count} tables")

    # Create detailed index file
    create_detailed_index(total_tables, category_counts)

def create_detailed_index(total_tables, category_counts):
    """Create a detailed index file listing all generated tables"""
    index_content = f"""ADVERSARIAL ANALYSIS TABLES INDEX
===========================================
Generated on: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
Total tables: {total_tables}
Location: {OUTPUT_DIR}

TABLE CATEGORIES AND COUNTS:
1.  Baseline Model Performance: {category_counts.get('Baseline', 0)} tables
2.  Adversarial Attack Effectiveness: {category_counts.get('Attack Effectiveness', 0)} tables
3.1 Adversarial Training Defense: {category_counts.get('Adversarial Training', 0)} tables
3.2 Randomized Smoothing Defense: {category_counts.get('Randomized Smoothing', 0)} tables
4.  Attack Transferability Analysis: {category_counts.get('Attack Transferability', 0)} tables
5.  Defense Comparison: {category_counts.get('Defense Comparison', 0)} tables

DETAILED LIST OF TABLES:
===========================================

1. BASELINE MODEL PERFORMANCE (3 tables):
-------------------------------------------
   - 1_baseline_performance_CSECICIDS2018.csv
   - 1_baseline_performance_TONIOT.csv
   - 1_baseline_performance_WUSTLEHMS2020.csv

2. ADVERSARIAL ATTACK EFFECTIVENESS (13 tables):
-------------------------------------------------
   2.1 FGSM Attack Effectiveness (3 tables):
   - 2_fgsm_effectiveness_CSECICIDS2018.csv
   - 2_fgsm_effectiveness_TONIOT.csv
   - 2_fgsm_effectiveness_WUSTLEHMS2020.csv

   2.2 PGD Attack Effectiveness (3 tables):
   - 3_pgd_effectiveness_CSECICIDS2018.csv
   - 3_pgd_effectiveness_TONIOT.csv
   - 3_pgd_effectiveness_WUSTLEHMS2020.csv

   2.3 CW Attack Effectiveness (3 tables):
   - 4_cw_effectiveness_CSECICIDS2018.csv
   - 4_cw_effectiveness_TONIOT.csv
   - 4_cw_effectiveness_WUSTLEHMS2020.csv

   2.4 Cross-Dataset Analysis (4 tables):
   - 5_mean_asr_across_datasets.csv
   - 6_mean_accuracy_FGSM_across_datasets.csv
   - 6_mean_accuracy_PGD_across_datasets.csv
   - 6_mean_accuracy_CW_across_datasets.csv

3. DEFENSE MECHANISM EVALUATION (58 tables):
---------------------------------------------
   3.1 Adversarial Training Defense (29 tables):
   - For each dataset (CSECICIDS2018, TONIOT, WUSTLEHMS2020):
     * at_effectiveness_[dataset].csv
     * at_fgsm_epsilon_FGSM_[dataset].csv
     * at_fgsm_epsilon_PGD_[dataset].csv
     * at_fgsm_ratio_FGSM_[dataset].csv
     * at_fgsm_ratio_PGD_[dataset].csv
     * at_pgd_epsilon_FGSM_[dataset].csv
     * at_pgd_epsilon_PGD_[dataset].csv
     * at_pgd_maxiter_FGSM_[dataset].csv
     * at_pgd_maxiter_PGD_[dataset].csv
     * at_pgd_ratio_FGSM_[dataset].csv
     * at_pgd_ratio_PGD_[dataset].csv
     * at_cw_confidence_FGSM_[dataset].csv
     * at_cw_confidence_PGD_[dataset].csv
     * at_cw_maxiter_FGSM_[dataset].csv
     * at_cw_maxiter_PGD_[dataset].csv
     * at_cw_ratio_FGSM_[dataset].csv
     * at_cw_ratio_PGD_[dataset].csv

   3.2 Randomized Smoothing Defense (29 tables):
   - For each dataset (CSECICIDS2018, TONIOT, WUSTLEHMS2020):
     * rs_certified_accuracy_[dataset].csv
     * rs_fgsm_epsilon_[dataset].csv
     * rs_fgsm_sigma_[dataset].csv
     * rs_pgd_epsilon_[dataset].csv
     * rs_pgd_maxiter_[dataset].csv
     * rs_pgd_sigma_[dataset].csv
     * rs_cw_confidence_[dataset].csv
     * rs_cw_maxiter_[dataset].csv
     * rs_cw_sigma_[dataset].csv

4. ATTACK TRANSFERABILITY ANALYSIS (4 tables):
-----------------------------------------------
   - 7_attack_transferability_CSECICIDS2018.csv
   - 7_attack_transferability_TONIOT.csv
   - 7_attack_transferability_WUSTLEHMS2020.csv
   - 8_overall_attack_transferability.csv

5. DEFENSE COMPARISON (1 table):
---------------------------------
   - 9_defense_methods_comparison.csv

NOTES:
- All values are formatted to 4 decimal places
- Model order: {', '.join(MODEL_ORDER)}
- Dataset order: {', '.join(DATASET_ORDER)}
- Attack order: {', '.join(ATTACK_ORDER)}
- File naming convention: [category_number]_[description]_[dataset/type].csv
"""

    index_path = os.path.join(OUTPUT_DIR, "TABLE_INDEX_DETAILED.txt")
    with open(index_path, 'w') as f:
        f.write(index_content)

    print(f"\nCreated detailed table index: {index_path}")

if __name__ == "__main__":
    main()

ADVERSARIAL ANALYSIS TABLE GENERATOR

Loading results files...
Loaded: base_results_CSECICIDS2018.csv
Loaded: attack_results_CSECICIDS2018.csv
Loaded: defense_results_CSECICIDS2018.csv

Found results for datasets: ['CSECICIDS2018']

GENERATING ALL 79 TABLES

1. Generating Baseline Model Performance Tables (3 tables)...
Saved: 1_baseline_performance_CSECICIDS2018.csv

2. Generating Adversarial Attack Effectiveness Tables (13 tables)...
Saved: 2_fgsm_effectiveness_CSECICIDS2018.csv
Saved: 3_pgd_effectiveness_CSECICIDS2018.csv
Saved: 4_cw_effectiveness_CSECICIDS2018.csv
  Generating cross-dataset tables...
Saved: 5_mean_asr_across_datasets.csv
Saved: 6_mean_accuracy_FGSM_across_datasets.csv
Saved: 6_mean_accuracy_PGD_across_datasets.csv
Saved: 6_mean_accuracy_CW_across_datasets.csv

3.1 Generating Adversarial Training Defense Tables (29 tables)...

  Processing CSECICIDS2018...
  Generating Table 1 for CSECICIDS2018
Saved: at_effectiveness_CSECICIDS2018.csv
  Generating Tables 2-3 for CSE